# Model precision characterisation

Two experiments:
1. **Noise floor** — how variable is the PSTH across stochastic runs? Features smaller than this can't be reliably targeted by waveform optimisation.
2. **Temporal resolution** — how precisely does a brief input pulse translate to a PSTH response, given membrane dynamics and network recurrence?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from designer_waveform.waveforms import (
    AsymBaselineSplitGaussianWaveform,
    RectangularPulseWaveform,
)
from designer_waveform.models import RandomEINetwork, load_config

In [ ]:
# ── Path to a saved optimisation result (from 06_05_26_optimise_splitgaussian_ei_snn_real_psth) ──
OPT_RESULT_PATH = Path('../results/21_05_26_siegle_fig4d_all_optimised_waveform/own_pipeline_all_optimisation_result.pkl')

# ── Network settings — match those used during optimisation ──────────────
N_EXC     = 2000
N_INH     = 500
T_PRE_MS  = 200.0
T_POST_MS = 100.0

# ── Run settings ─────────────────────────────────────────────────────────
N_RUNS      = 20
SEED_BASE   = 1000
BIN_SIZE_MS = 10.0

## 1. Noise floor

Run the optimised waveform `N_RUNS` times with independent seeds and measure the per-bin standard deviation.
Features in the target PSTH with amplitude < ~2σ are below the noise floor and cannot be reliably fit.

In [ ]:
with open(OPT_RESULT_PATH, 'rb') as f:
    saved = pickle.load(f)

opt_waveform = saved['opt_waveform']
target_hz    = saved['target_hz']
target_err   = saved['target_err']
target_t_ms  = saved['target_t_ms']
STIM_DUR_MS  = float(saved['stim_dur_ms'])

CONFIG_PATH = Path('..') / 'configs' / 'random_ei.json'
cfg = load_config(CONFIG_PATH)
cfg.N_exc       = N_EXC
cfg.N_inh       = N_INH
cfg.t_pre_ms    = T_PRE_MS
cfg.t_post_ms   = T_POST_MS
cfg.t_stim_ms   = STIM_DUR_MS
cfg.psth_bin_ms = BIN_SIZE_MS

model = RandomEINetwork(cfg)
print(f'Optimised waveform : {opt_waveform}')
print(f'Stim duration      : {STIM_DUR_MS:.0f} ms')
print(f'Bin size           : {BIN_SIZE_MS:.0f} ms')

In [ ]:
_psths = []
print(f'Running {N_RUNS} simulations for noise floor...')
for _i in range(N_RUNS):
    _r = model.run(opt_waveform, seed=SEED_BASE + _i)
    _psths.append(_r['psth_exc'] / (BIN_SIZE_MS / 1000.0))
    if (_i + 1) % 5 == 0:
        print(f'  {_i + 1}/{N_RUNS}')

_psth_arr = np.stack(_psths)
_t_psth   = model.run(opt_waveform)['t_psth_ms']

noise_mean = _psth_arr.mean(0)
noise_std  = _psth_arr.std(0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
if not np.isnan(target_err).all():
    ax.fill_between(target_t_ms, target_hz - target_err, target_hz + target_err,
                    color='k', alpha=0.15)
ax.plot(target_t_ms, target_hz, 'k', lw=1.8, label='Target')
ax.fill_between(_t_psth, noise_mean - 2*noise_std, noise_mean + 2*noise_std,
                color='tomato', alpha=0.25, label='SNN mean \u00b12\u03c3')
ax.plot(_t_psth, noise_mean, color='tomato', lw=1.8, label=f'SNN mean (n={N_RUNS})')
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Noise floor vs target')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.bar(_t_psth, noise_std, width=BIN_SIZE_MS * 0.9, color='steelblue', alpha=0.7,
       label='\u03c3 across runs (SNN)')
if not np.isnan(target_err).all():
    ax.bar(target_t_ms, target_err, width=BIN_SIZE_MS * 0.9, color='k', alpha=0.3,
           label='SEM (Allen data)')
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Std dev (Hz)')
ax.set_title('Per-bin noise magnitude')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()

print(f'\nMean \u03c3 across bins : {noise_std.mean():.2f} Hz')
print(f'Peak \u03c3             : {noise_std.max():.2f} Hz at t={_t_psth[noise_std.argmax()]:.0f} ms')
print(f'\nFeatures in the target smaller than {2*noise_std.mean():.1f} Hz are likely below the noise floor.')

## 2. Temporal resolution — pulse sweep

Apply a brief rectangular probe pulse at different onset times across the stim window.
For each position, run `N_RUNS_SWEEP` simulations and compute the mean PSTH.

The width of the PSTH response characterises how much temporal blurring the membrane dynamics and network recurrence introduce.
The diagonal offset between pulse onset and PSTH peak characterises response latency.

In [ ]:
PULSE_WIDTH_MS = 10.0   # width of probe pulse (ms)
PULSE_AMP      = 0.3    # envelope amplitude — should evoke a clear response
SWEEP_STEP_MS  = 10.0   # spacing between probe onset positions
N_RUNS_SWEEP   = 10     # runs per position (fewer than noise floor — many positions)

pulse_onsets = np.arange(SWEEP_STEP_MS, STIM_DUR_MS - PULSE_WIDTH_MS, SWEEP_STEP_MS)
psth_per_onset = []

print(f'Sweeping {PULSE_WIDTH_MS:.0f} ms pulse at {len(pulse_onsets)} positions...')
for _onset in pulse_onsets:
    wf = RectangularPulseWaveform(onset_ms=_onset, duration_ms=PULSE_WIDTH_MS,
                                   amplitude=PULSE_AMP)
    _runs = []
    for _i in range(N_RUNS_SWEEP):
        _r = model.run(wf, seed=SEED_BASE + _i)
        _runs.append(_r['psth_exc'] / (BIN_SIZE_MS / 1000.0))
    psth_per_onset.append(np.stack(_runs).mean(0))
    print(f'  onset={_onset:.0f} ms', end='\r')

psth_per_onset = np.stack(psth_per_onset)  # (n_onsets, n_bins)
print(f'\nDone. Array shape: {psth_per_onset.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap: rows = pulse onset, cols = PSTH time
ax = axes[0]
im = ax.imshow(
    psth_per_onset,
    extent=[_t_psth[0], _t_psth[-1], pulse_onsets[-1] + SWEEP_STEP_MS/2,
            pulse_onsets[0] - SWEEP_STEP_MS/2],
    aspect='auto', cmap='hot', interpolation='nearest',
)
ax.plot([_t_psth[0], _t_psth[-1]], [_t_psth[0], _t_psth[-1]],
        'w--', lw=0.8, label='pulse onset (diag.)')
plt.colorbar(im, ax=ax, label='Firing rate (Hz)')
ax.set_xlabel('Time in stim window (ms)')
ax.set_ylabel('Pulse onset (ms)')
ax.set_title(f'Pulse sweep heatmap ({PULSE_WIDTH_MS:.0f} ms pulse, amp={PULSE_AMP})')
ax.legend(frameon=False, fontsize=7)

# Overlaid traces coloured by onset position
ax = axes[1]
_cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(pulse_onsets)))
for psth, color in zip(psth_per_onset, _cmap):
    ax.plot(_t_psth, psth, color=color, lw=1.0, alpha=0.8)
sm = plt.cm.ScalarMappable(cmap='viridis',
     norm=plt.Normalize(pulse_onsets.min(), pulse_onsets.max()))
plt.colorbar(sm, ax=ax, label='Pulse onset (ms)')
ax.set_xlabel('Time in stim window (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('PSTH response per pulse position')
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# For each pulse position: measure peak delay and PSTH half-width
_floor = noise_std.mean()  # use noise-floor sigma as detection threshold
_delays, _widths, _valid_onsets = [], [], []

for psth, onset in zip(psth_per_onset, pulse_onsets):
    _pre_idx = max(1, np.searchsorted(_t_psth, onset) - 1)
    _bg = psth[:_pre_idx].mean() if _pre_idx > 0 else 0.0
    _above = psth - _bg
    _peak = _above.max()
    if _peak < 2 * _floor:
        continue   # below noise floor — skip
    _half = _peak / 2.0
    _idxs = np.where(_above >= _half)[0]
    _delays.append(float(_t_psth[_above.argmax()]) - onset)
    _widths.append(float(_t_psth[_idxs[-1]] - _t_psth[_idxs[0]]) if len(_idxs) > 1 else float(BIN_SIZE_MS))
    _valid_onsets.append(onset)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

ax = axes[0]
ax.plot(_valid_onsets, _delays, 'o-', color='steelblue', lw=1.5)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Pulse onset (ms)')
ax.set_ylabel('Peak response delay (ms)')
ax.set_title('Response latency vs pulse position')
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.plot(_valid_onsets, _widths, 's-', color='tomato', lw=1.5)
ax.axhline(BIN_SIZE_MS, color='grey', lw=0.8, ls='--', label=f'bin size ({BIN_SIZE_MS:.0f} ms)')
ax.set_xlabel('Pulse onset (ms)')
ax.set_ylabel('PSTH half-width (ms)')
ax.set_title('Temporal response width vs pulse position')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()

if _widths:
    print(f'Mean response latency  : {np.mean(_delays):.1f} ms')
    print(f'Mean PSTH half-width   : {np.mean(_widths):.1f} ms  (temporal resolution estimate)')
    print(f'\nFeatures narrower than ~{np.mean(_widths):.0f} ms in the target are likely unresolvable.')